# Audio Signal Analysis & Sound Classification (Part 2)

In this notebook, we will continue the work from yesterday, but now focusing on classification of audio events
- Extract audio features
- Build an ML model that **classifies environmental sounds**


Remember: Change the venv to `sted-workshop-venv`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path
import pandas as pd

In [ ]:
# Extracting the environment sound...
import os, zipfile, urllib.request

def download_and_extract_esc50():
    esc50_dir = './ESC-50-master'

    if not os.path.exists(esc50_dir):
        print("Downloading ESC-50 dataset (this takes ~1 minute)...")
        url = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"
        zip_path = "ESC-50.zip"
        urllib.request.urlretrieve(url, zip_path)
        
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('.')
        os.remove(zip_path)
        print("Done!")
    else:
        print("ESC-50 already downloaded.")

    # Load the metadata
    meta = pd.read_csv(os.path.join(esc50_dir, 'meta', 'esc50.csv'))
    print(f"\nTotal clips: {len(meta)}")
    print(f"Categories: {meta['category'].nunique()}")
    print(f"\nAll categories:")
    for i, cat in enumerate(sorted(meta['category'].unique()), 1):
        print(f"  {i:2d}. {cat}")

    return esc50_dir, meta

esc50_dir, meta = download_and_extract_esc50()

## Part 4: Extracting audio features


In [ ]:
# Extract features from the trumpet clip
def extract_features(y, sr):
    """Extract audio features from a signal."""
    features = {}
    
    # MFCCs — 13 coefficients, averaged over time
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i+1}'] = np.mean(mfccs[i])
    
    # Spectral centroid
    features['spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    
    # Zero-crossing rate
    features['zero_crossing_rate'] = np.mean(librosa.feature.zero_crossing_rate(y))
    
    # Spectral bandwidth
    features['spectral_bandwidth'] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    
    # RMS energy
    features['rms_energy'] = np.mean(librosa.feature.rms(y=y))
    
    # Spectral rolloff
    features['spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    
    return features

# Extract features from our two instruments
y_trumpet, sr_trumpet = librosa.load(librosa.ex('trumpet'))
y_piano, sr_piano = librosa.load(librosa.ex('pistachio'))
trumpet_features = extract_features(y_trumpet, sr_trumpet)
piano_features = extract_features(y_piano, sr_piano)

# Compare
import pandas as pd
comparison = pd.DataFrame({
    'Trumpet': trumpet_features,
    'Piano': piano_features
})

# Show the non-MFCC features for clarity
print("Feature comparison (non-MFCC):")
print(comparison.loc[~comparison.index.str.startswith('mfcc')].round(2))
print("\nMFCC comparison:")
print(comparison.loc[comparison.index.str.startswith('mfcc')].round(2))

### Visualizing MFCCs over time

MFCCs are the most important features in audio ML. Let's see what they look like as a time series:

In [ ]:
# MFCCs over time for trumpet vs piano
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

mfccs = librosa.feature.mfcc(y=y_trumpet, sr=sr_trumpet, n_mfcc=13)
img1 = librosa.display.specshow(mfccs, sr=sr_trumpet, x_axis='time', ax=axes[0])
axes[0].set_title('MFCCs over Time')
axes[0].set_ylabel('MFCC Coefficient')
fig.colorbar(img1, ax=axes[0])

## Part 5: Building a Sound Event Classifier

### The ML pipeline

```
1. Load audio files
2. Extract features from each clip
3. Train/test split
4. Train classifier
5. Evaluate with confusion matrix
```

In [ ]:
esc50_dir, meta = download_and_extract_esc50()

In [ ]:
# Select 10 interesting categories for our classifier
selected_categories = [
    'dog', 'rain', 'clock_tick', 'siren', 'clapping',
    'helicopter', 'sea_waves', 'sneezing', 'engine', 'cat'
]

meta_subset = meta[meta['category'].isin(selected_categories)].copy()
print(f"Selected {len(meta_subset)} clips across {len(selected_categories)} categories")
print(meta_subset['category'].value_counts())

In [ ]:
# Play one example from each category
audio_dir = os.path.join(esc50_dir, 'audio')

fig, axes = plt.subplots(2, 5, figsize=(18, 5))
axes = axes.flatten()

for i, category in enumerate(selected_categories):
    # Get the first clip in this category
    row = meta_subset[meta_subset['category'] == category].iloc[0]
    filepath = os.path.join(audio_dir, row['filename'])
    
    y, sr = librosa.load(filepath, sr=22050)
    
    # Plot waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[i])
    axes[i].set_title(category, fontsize=10)
    axes[i].set_xlabel('')
    
    # Display audio player (only for first 5 to save space)
    if i < 5:
        print(f"--- {category} ---")
        ipd.display(ipd.Audio(y, rate=sr))

plt.tight_layout()
plt.show()

In [ ]:
# Extract features from all selected clips
audio_dir = os.path.join(esc50_dir, 'audio')

all_features = []
all_labels = []
failed = 0

print(f"Extracting features from {len(meta_subset)} clips...")
for idx, (_, row) in enumerate(meta_subset.iterrows()):
    filepath = os.path.join(audio_dir, row['filename'])
    try:
        y, sr = librosa.load(filepath, sr=22050, duration=5.0)
        feats = extract_features(y, sr)
        all_features.append(feats)
        all_labels.append(row['category'])
    except Exception as e:
        failed += 1
    
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(meta_subset)} clips...")

# Create DataFrame
features_df = pd.DataFrame(all_features)
features_df['category'] = all_labels

print(f"\nDone! Extracted features from {len(features_df)} clips ({failed} failed)")
print(f"Feature matrix shape: {features_df.shape}")
features_df.head()

### Implementing RandomForestClassifier

In [ ]:
# Prepare for ML

In [ ]:
# Separating features and labels (We will use features to predict labels)

In [ ]:
# Train/test split
## Try with sklearn.model_selection.train_test_split

In [ ]:
# Check: What is in each variable, and what do they mean? Discuss with your neighbors

In [ ]:
# Train a Random Forest classifier
## Try with sklearn.ensemble.RandomForestClassifier
rf_model = 0

In [ ]:
# Evaluate
## Try with sklearn.metrics.classification_report, accuracy_score

In [ ]:
# Confusion matrix — shows exactly where the model gets confused
## Try with sklearn.metrics.confusion_matrix

In [ ]:
# Which features matter most?


## Part 6: Challenge: Classify Your Own Sounds!

### Either: Test on the data in the dataset, or your own sound!

In [ ]:
# Test on a random clip from the test set
import random

test_idx = random.randint(0, len(meta_subset) - 1)
test_row = meta_subset.iloc[test_idx]
test_file = os.path.join(audio_dir, test_row['filename'])

y_test_clip, sr_test = librosa.load(test_file, sr=22050, duration=5.0)

# Listen
print(f"True label: {test_row['category']}")
ipd.display(ipd.Audio(y_test_clip, rate=sr_test))

In [ ]:
# Extract features and predict
test_features = extract_features(y_test_clip, sr_test)
test_df = pd.DataFrame([test_features])
prediction = rf_model.predict(test_df)[0]
probabilities = rf_model.predict_proba(test_df)[0]

print(f"Predicted: {prediction}")
print(f"\nConfidence for each category:")
for cat, prob in sorted(zip(rf_model.classes_, probabilities), key=lambda x: -x[1]):
    bar = '#' * int(prob * 40)
    print(f"  {cat:15s} {prob:.1%} {bar}")